# EMR Analytics Showcase\n\nThis notebook presents a portfolio-ready exploratory analysis of a synthetic EMR-style dataset.\nIt combines SQL, pandas, and visualization to review patient demographics, appointment operations, billing performance, treatment-note activity, and abnormal lab results.\n\nThe goal is to show how a healthcare-flavoured relational dataset can be explored from both an operational and analytical perspective.\n

In [ ]:
import sqlite3\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom pathlib import Path\n\nsns.set_theme(style='whitegrid')\npd.options.display.float_format = '{:,.2f}'.format\n\ndef find_db_path() -> Path:\n    candidates = [Path.cwd(), *Path.cwd().parents]\n    for base in candidates:\n        db_path = (base / 'data' / 'emr_showcase.db').resolve()\n        if db_path.exists():\n            return db_path\n    raise FileNotFoundError('Could not locate data/emr_showcase.db from the current working directory.')\n\nDB_PATH = find_db_path()\nconn = sqlite3.connect(DB_PATH)\nDB_PATH\n

In [ ]:
schema = pd.read_sql_query(\n    \"SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name\",\n    conn\n)\nschema[['name']]\n

## Executive Summary\n\nThis synthetic dataset is useful for answering a few practical questions quickly:\n- How large is the patient and appointment base?\n- What is the overall revenue picture?\n- How many abnormal lab results may require follow-up?\n- Which clinical areas generate the most treatment-note activity?\n

In [ ]:
overview = pd.DataFrame(\n    {\n        'metric': [\n            'Patients',\n            'Appointments',\n            'Paid revenue',\n            'Abnormal lab results'\n        ],\n        'value': [\n            pd.read_sql_query(\"SELECT COUNT(*) AS value FROM patients\", conn).iloc[0, 0],\n            pd.read_sql_query(\"SELECT COUNT(*) AS value FROM appointments\", conn).iloc[0, 0],\n            pd.read_sql_query(\"SELECT ROUND(COALESCE(SUM(amount), 0), 2) AS value FROM payments WHERE payment_status = 'Paid'\", conn).iloc[0, 0],\n            pd.read_sql_query(\"SELECT COUNT(*) AS value FROM lab_results WHERE result_flag = 'Abnormal'\", conn).iloc[0, 0],\n        ]\n    }\n)\noverview\n

## 1. Patient Demographics\n

In [ ]:
patients = pd.read_sql_query('SELECT * FROM patients', conn)\npatients.head()\n

In [ ]:
patients['risk_level'].value_counts().plot(kind='bar', color=['#4caf50', '#ffb300', '#e53935'])\nplt.title('Patients by Risk Level')\nplt.xlabel('Risk Level')\nplt.ylabel('Count')\nplt.tight_layout()\nplt.show()\n

## 2. Appointment Volumes and Status\n

In [ ]:
monthly_appointments = pd.read_sql_query(\n    \"\"\"\n    SELECT strftime('%Y-%m', start_time) AS month, appointment_type, COUNT(*) AS appointment_count\n    FROM appointments\n    GROUP BY month, appointment_type\n    ORDER BY month, appointment_type\n    \"\"\",\n    conn\n)\nmonthly_appointments.head()\n

In [ ]:
pivot = monthly_appointments.pivot(index='month', columns='appointment_type', values='appointment_count').fillna(0)\npivot.plot(figsize=(12, 5), marker='o')\nplt.title('Monthly Appointment Volume by Type')\nplt.xlabel('Month')\nplt.ylabel('Appointments')\nplt.xticks(rotation=45)\nplt.tight_layout()\nplt.show()\n

## 3. Billing and Revenue\n

In [ ]:
revenue = pd.read_sql_query(\n    \"\"\"\n    SELECT payment_status, ROUND(SUM(amount), 2) AS total_amount, COUNT(*) AS invoice_count\n    FROM payments\n    GROUP BY payment_status\n    ORDER BY total_amount DESC\n    \"\"\",\n    conn\n)\nrevenue\n

In [ ]:
sns.barplot(data=revenue, x='payment_status', y='total_amount')\nplt.title('Revenue by Payment Status')\nplt.xlabel('Payment Status')\nplt.ylabel('Total Amount')\nplt.tight_layout()\nplt.show()\n

## 4. Treatment Notes and Diagnosis Mix\n

In [ ]:
diagnosis_mix = pd.read_sql_query(\n    \"\"\"\n    SELECT diagnosis_group, ROUND(AVG(note_length), 1) AS avg_note_length, COUNT(*) AS note_count\n    FROM treatment_notes\n    GROUP BY diagnosis_group\n    ORDER BY note_count DESC\n    \"\"\",\n    conn\n)\n\nplt.figure(figsize=(10, 5))\nsns.barplot(\n    data=diagnosis_mix.sort_values('note_count', ascending=False),\n    x='note_count',\n    y='diagnosis_group',\n    hue='diagnosis_group',\n    palette='Blues_r',\n    legend=False\n)\nplt.title('Treatment Notes by Diagnosis Group')\nplt.xlabel('Number of Notes')\nplt.ylabel('Diagnosis Group')\nplt.tight_layout()\nplt.show()\n\ndiagnosis_mix\n

## 5. Abnormal Lab Results\n

In [ ]:
abnormal_labs = pd.read_sql_query(\n    \"\"\"\n    SELECT p.patient_code, p.full_name, p.age, l.test_name, l.result_value, l.result_flag, l.encounter_date\n    FROM lab_results l\n    JOIN patients p ON p.id = l.patient_id\n    WHERE l.result_flag = 'Abnormal'\n    ORDER BY l.encounter_date DESC\n    \"\"\",\n    conn\n)\nabnormal_labs.head(20)\n

## 6. Example Join Query for Portfolio Discussion\n\nThis query demonstrates how to join patient, appointment, and payment data to discuss utilization and payment workflow in interviews.\n

In [ ]:
patient_utilization = pd.read_sql_query(\n    \"\"\"\n    SELECT\n        p.patient_code,\n        p.full_name,\n        COALESCE(a.total_appointments, 0) AS total_appointments,\n        COALESCE(t.total_notes, 0) AS total_notes,\n        ROUND(COALESCE(pay.paid_revenue, 0), 2) AS paid_revenue\n    FROM patients p\n    LEFT JOIN (\n        SELECT patient_id, COUNT(*) AS total_appointments\n        FROM appointments\n        GROUP BY patient_id\n    ) a ON a.patient_id = p.id\n    LEFT JOIN (\n        SELECT patient_id, COUNT(*) AS total_notes\n        FROM treatment_notes\n        GROUP BY patient_id\n    ) t ON t.patient_id = p.id\n    LEFT JOIN (\n        SELECT patient_id, SUM(amount) AS paid_revenue\n        FROM payments\n        WHERE payment_status = 'Paid'\n        GROUP BY patient_id\n    ) pay ON pay.patient_id = p.id\n    ORDER BY total_appointments DESC, paid_revenue DESC\n    LIMIT 15\n    \"\"\",\n    conn\n)\npatient_utilization\n

## Portfolio Talking Points\n\n- Synthetic EMR schema covering patients, appointments, treatment notes, payments, and lab results\n- SQL used for joins, aggregations, and operational reporting\n- Notebook used to surface patient-risk mix, appointment trends, revenue, and follow-up signals\n- Clear example of translating healthcare-style relational data into recruiter-friendly analysis outputs\n

In [ ]:
conn.close()\n